import needed libraries

In [1]:
from ipyleaflet import Marker, Map, GeoData, Polyline, AwesomeIcon
from pyrosm import get_data, OSM
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString
import networkx as nx

get Tehran driving road map from OSM attribut of pyrosm and then change it to geodataframe with geopandas library

In [2]:
pbf = get_data("Tehran")
osm = OSM(pbf)
roads = osm.get_network("driving")
roads_frame = gpd.GeoDataFrame(roads, geometry="geometry").copy()
roads_frame = roads_frame[~roads_frame.geometry.is_empty]

c:\Users\navid\anaconda3\Lib\site-packages\pyrosm\networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(


load Tehran general map with ipyleaflet and make two different markers and stick them to the map by add_layer()

In [3]:

tehran_map = Map(center=(35.6892, 51.3890), zoom=12)


marker1 = Marker(location=(35.6892, 51.3890), draggable=True, visible=False)
marker2 = Marker(location=(35.7, 51.4), draggable=True, visible=False)


tehran_map.add_layer(marker1)
tehran_map.add_layer(marker2)


we made layers that is all roads layer because we want to stick theas roads_frame to the general map if the map is not empty 

In [4]:
if not roads_frame.empty:
    layers = GeoData(
        geo_dataframe=roads_frame,
        style={'color': 'gray', 'weight': 1.5},
        name="driving roads"
    )
    tehran_map.add_layer(layers)

know if we wanna work on our road map we have to make it a grapg with edges and nodes. 
what does to_graph def do??
first it makes an empty graph
it gets an entry that is our road data frame and then it itterates over its rows and gets each row geometry that is needed for making the graph
then it evaluates that if the data is a multilinestrin(A multiline string graph (or multiline graph) is a type of data visualization that displays multiple lines on the same plot, where each line represents a different category, group, or time series. This allows for easy comparison between different datasets.)
and if it is multiline we change it to a single line string.
then we itterate over those single lines we made.
we extract coords those are sequence (x,y) , and after that we find distances between each two edges and with add_edge attribute we add edges and return made graph


In [5]:

def to_graph(roads_gdf):
    G = nx.Graph()
    for _, row in roads_gdf.iterrows():
        geometries = row.geometry
        if isinstance(geometries, MultiLineString):
            lines = geometries.geoms
        elif isinstance(geometries, LineString):
            lines = [geometries]
        else:
            continue

        for line in lines:
            coords = list(line.coords)
            for i in range(len(coords) - 1):
                u = coords[i]
                v = coords[i + 1]
                dist = Point(u).distance(Point(v))
                G.add_edge(u, v, weight=dist)
    return G

G = to_graph(roads_frame)

def nearst_grapg_node findes nearest node  with that point we pass to it(haer is the marker). target is each point that is passed and it will be campared with each node distance to the marker and the minimun will be returned.

In [6]:
def nearest_graph_node(point, G):
    target = Point(point[1], point[0])  
    nearest = min(G.nodes, key=lambda n: target.distance(Point(n)))
    return nearest

the most important part of the first part of project is updatae_way def. by defult we have no way so our variable (way) is None
if we have any way and we wanna update our way, first we have to remove the way layer that has been made before.
we wanna draw shortest way(path) between those two marker,so they have to be visiable.
if both of them are made(visiable), we find nearest node to both of them and put them in start and end variable.
and the shortest path will be found with shortest_path attribute of networkx library and it will be put in a layer and will be add to the Tehran map

In [7]:

way = None

def update_way():
    global way
    if way:
        tehran_map.remove_layer(way)
        way = None

    if marker1.visible and marker2.visible:
        start = nearest_graph_node(marker1.location, G)
        end = nearest_graph_node(marker2.location, G)


        path = nx.shortest_path(G, source=start, target=end)
        path_coords = [(y, x) for x, y in path]

        way = Polyline(
            locations=path_coords,
            color="blue",
            weight=5,
            opacity=0.8
        )
        tehran_map.add_layer(way)

click_handler is a simple def that has been bined to "click" and it gets the latitute and longitude and it will make the first and second marker location and make them visiable and each time runs update_way def

In [8]:
click_count = 0
def click_handler(**kwargs):
    global click_count
    if kwargs.get('type') == 'click':
        lat, lon = kwargs.get('coordinates')

        if click_count % 2 == 0:
            marker1.location = (lat, lon)
            marker1.visible = True
        else:
            marker2.location = (lat, lon)
            marker2.visible = True

        click_count += 1
        update_way()

and the last part fits the roads with the general map. rerange the map in a new format that is a list filed with tuples and fit the sections with fit_bounds onto the map

In [ ]:
tehran_map.on_interaction(click_handler)

if not roads_frame.empty:
    sections = roads_frame.total_bounds
    map_bounds = [(sections[1], sections[0]), (sections[3], sections[2])]
    tehran_map.fit_bounds(map_bounds)

tehran_map


Map(center=[35.6892, 51.389], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [6]:
pois = osm.get_pois()
unis = pois[(pois["amenity"]=="university") & (pois["name"].notna())& (pois.geometry.type == "Point")]


for _, row in unis.iterrows():
    if row.geometry.geom_type == "Point":
        lat = row.geometry.y
        lon = row.geometry.x
        uni_marker = AwesomeIcon(name="universities",marker_color="red",icon_color='white')
        marker = Marker(location=(lat,lon),icon = uni_marker ,dragable = False)
        tehran_map.add_layer(marker)



tehran_map        

Map(bottom=1651182.0, center=[35.72602928962275, 51.50802612304688], controls=(ZoomControl(options=['position'…